# Kernel SHAP

モデル非依存（model-agnostic）、つまり任意の機械学習アルゴリズムで作った予測モデルに対して適用できるSHAP values推定アルゴリズム。

先行研究にはLIMEという局所線形近似によって説明モデルを作る手法がある。LIME推定量がShapley valueになるかどうかは損失関数L、重みカーネルπ、正則化項Ωに依存する。  
**以下のΩ, π, Lを使えば推定量がShapley valueになる**。


:::{admonition} Theorem 2 (Shapley kernel)

$$
\begin{aligned}
\Omega(g) & =0 \\
\pi_{x^{\prime}}\left(z^{\prime}\right) & =\frac{(M-1)}{\left(M \text { choose }\left|z^{\prime}\right|\right)\left|z^{\prime}\right|\left(M-\left|z^{\prime}\right|\right)},
\\
L\left(f, g, \pi_{x^{\prime}}\right) & =\sum_{z^{\prime} \in Z}\left[f\left(h_x\left(z^{\prime}\right)\right)-g\left(z^{\prime}\right)\right]^2 \pi_{x^{\prime}}\left(z^{\prime}\right),
\end{aligned}
$$

のもとで、LIMEの推定量

$$
\xi=\operatorname*{\arg \min}_{g \in \mathcal{G}} L\left(f, g, \pi_{x^{\prime}}\right)+\Omega(g)
$$

は性質1~3を満たす。ここで$|z'|$は$z'$の非ゼロ要素の数。
:::

誤差関数は重みつきの二乗誤差　→　重み付き最小二乗法で推定できる。

[Simple Kernel SHAP — SHAP latest documentation](https://shap.readthedocs.io/en/latest/example_notebooks/tabular_examples/model_agnostic/Simple%20Kernel%20SHAP.html)

In [2]:
import itertools

import numpy as np
import scipy.special


def powerset(iterable):
    s = list(iterable)
    return itertools.chain.from_iterable(itertools.combinations(s, r) for r in range(len(s) + 1))


def shapley_kernel(M, s):
    if s == 0 or s == M:
        return 10000
    return (M - 1) / (scipy.special.binom(M, s) * s * (M - s))


def f(X):
    np.random.seed(0)
    beta = np.random.rand(X.shape[-1])
    return np.dot(X, beta) + 10


def kernel_shap(f, x, reference, M):
    X = np.zeros((2**M, M + 1))
    X[:, -1] = 1
    weights = np.zeros(2**M)
    V = np.zeros((2**M, M))
    for i in range(2**M):
        V[i, :] = reference

    ws = {}
    for i, s in enumerate(powerset(range(M))):
        s = list(s)
        V[i, s] = x[s]
        X[i, s] = 1
        ws[len(s)] = ws.get(len(s), 0) + shapley_kernel(M, len(s))
        weights[i] = shapley_kernel(M, len(s))
    y = f(V)
    wsq = np.sqrt(weights)
    result = np.linalg.lstsq(wsq[:, None] * X, wsq * y, rcond=None)[0]
    return result


M = 4
np.random.seed(1)
x = np.random.randn(M)
reference = np.zeros(M)
phi = kernel_shap(f, x, reference, M)
base_value = phi[-1]
shap_values = phi[:-1]

print("  reference =", reference)
print("          x =", x)
print("shap_values =", shap_values)
print(" base_value =", base_value)
print("   sum(phi) =", np.sum(phi))
print("       f(x) =", f(x))

  reference = [0. 0. 0. 0.]
          x = [ 1.62434536 -0.61175641 -0.52817175 -1.07296862]
shap_values = [ 0.89146267 -0.43752168 -0.31836259 -0.58464256]
 base_value = 10.000000000000002
   sum(phi) = 9.550935842131224
       f(x) = 9.55093584213122


In [15]:
M = 4 # 特徴量の数
np.random.seed(1)
x = np.random.randn(M)

def f(x):
    """線形回帰モデル"""
    np.random.seed(0)
    beta = np.random.rand(x.shape[-1])  # 回帰係数（乱数）
    return x @ beta + 10

f(x)

9.55093584213122

重みカーネル

$$
\pi_{x^{\prime}}\left(z^{\prime}\right)
=\frac{(M-1)}{\left(M \text { choose }\left|z^{\prime}\right|\right)\left|z^{\prime}\right|\left(M-\left|z^{\prime}\right|\right)}
$$

In [16]:
import scipy.special

def shapley_kernel(M, s):
    if s == 0 or s == M:
        return 10000
    return (M - 1) / (scipy.special.binom(M, s) * s * (M - s))

In [ ]:
import itertools

import numpy as np
import scipy.special


def powerset(iterable):
    s = list(iterable)
    return itertools.chain.from_iterable(itertools.combinations(s, r) for r in range(len(s) + 1))



def kernel_shap(f, x, reference, M):
    X = np.zeros((2**M, M + 1))
    X[:, -1] = 1
    weights = np.zeros(2**M)
    V = np.zeros((2**M, M))
    for i in range(2**M):
        V[i, :] = reference

    ws = {}
    for i, s in enumerate(powerset(range(M))):
        s = list(s)
        V[i, s] = x[s]
        X[i, s] = 1
        ws[len(s)] = ws.get(len(s), 0) + shapley_kernel(M, len(s))
        weights[i] = shapley_kernel(M, len(s))
    y = f(V)
    wsq = np.sqrt(weights)
    result = np.linalg.lstsq(wsq[:, None] * X, wsq * y, rcond=None)[0]
    return result


base_value = phi[-1]
shap_values = phi[:-1]

print("  reference =", reference)
print("          x =", x)
print("shap_values =", shap_values)
print(" base_value =", base_value)
print("   sum(phi) =", np.sum(phi))
print("       f(x) =", f(x))